In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path
import numpy as np


In [2]:
PROJECT_ROOT = Path().resolve().parent

In [ ]:
sample_fp = PROJECT_ROOT / "outputs" / "sample_points_indices.csv"
samples = pd.read_csv(sample_fp)

samples = samples.dropna(subset="lc")

pred_cols = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B8', 'NDVI', 'NDWI', 'MNDWI', 'NDCI', 'NDMI', 'NDBI', 'NDAVI']
metadata_cols = ["label_id","location","lc","class_int", "obs_date"]

# train_samples = samples.loc[(samples["location"]== "hartbeesport") | (samples["location"]== "rawapening")]
# test_samples = samples.loc[samples["location"]=="mula"]


y = samples['lc'].values
X = samples[pred_cols].values
metadata = samples[metadata_cols]



In [18]:
X.head()

AttributeError: 'numpy.ndarray' object has no attribute 'head'

### Criterion and scoring in RF and GroupKFold

- *criterion* in RandomForestClassifier(criterion="gini") — this is a model hyperparameter, it controls how the trees are built during training
- *scoring* in cross_val_score(scoring="f1_macro") — this is an evaluation metric, it controls how the fitted model is assessed on the held-out fold

In [ ]:
outer_cv = LeaveOneGroupOut()
inner_cv = LeaveOneGroupOut()

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

groups = samples['location'].values # values gives you numpy array

outer_results = {
        "test_location":[],
         "f1": [],
        "f1_macro":[],
        "accuracy": [],
        "best_n_estimators": [],
        "best_max_depth": [],
        "best_min_samples_leaf":[],
        "train_f1": [],
        "train_f1_macro": [],
        "train_accuracy": []
        }

param_grid = {
    'model__n_estimators': [100,200,500], # have to prefix the keys with the pipeline stepname followed by __
    'model__max_depth': [3,5,10,20,None],
    'model__min_samples_leaf': [1, 5, 10]
    }

search = GridSearchCV(estimator = pipe, # the pipeline goes here, instead of model
                        param_grid = param_grid,
                        scoring = 'f1',
                        cv = inner_cv,
                        n_jobs= 1)
all_predictions = []

for train_idx, test_idx in outer_cv.split(X, y, groups = groups):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    groups_train = groups[train_idx]
    test_location = groups[test_idx][0]
    test_metadata = metadata.iloc[test_idx]
    
    
    search.fit(X_train, y_train, groups= groups_train) # note: dont assign search.fit, it modifies in place.

    best_model= search.best_estimator_

    # predict on validatation region

    y_pred = best_model.predict(X_test) # best model is now the pipeline and the transforms will be applied to the test data before (eg scaling)

    # want to also predict on the train regions so can asses overfitting
    train_pred = best_model.predict(X_train)
    train_f1 = f1_score(y_train, train_pred, pos_label= 1, average= "binary")
    train_f1_macro = f1_score(y_train, train_pred, average= "macro")
    train_accuracy = accuracy_score(y_train, train_pred)
    #------------------------------------------------------------
    # calculate and save the results of predicting to thetest region for this fold.``

    fold_f1 = f1_score(y_test, y_pred, pos_label= 1, average= "binary")
    fold_f1macro = f1_score(y_test, y_pred, average= "macro")
    fold_accuracy = accuracy_score(y_test, y_pred)
    fold_best_n_estimators = search.best_params_["model__n_estimators"]
    fold_best_max_depth = search.best_params_["model__max_depth"]
    fold_best_min_samples_leaf = search.best_params_["model__min_samples_leaf"]

    outer_results["test_location"].append(test_location)
    outer_results["f1"].append(fold_f1)
    outer_results["f1_macro"].append(fold_f1macro)
    outer_results["accuracy"].append(fold_accuracy)
    outer_results["best_n_estimators"].append(fold_best_n_estimators)
    outer_results["best_max_depth"].append(fold_best_max_depth)
    outer_results["best_min_samples_leaf"].append(fold_best_min_samples_leaf)
    outer_results["train_f1"].append(train_f1)
    outer_results["train_f1_macro"].append(train_f1_macro)
    outer_results["train_accuracy"].append(train_accuracy)
    
    #------------------------------------------------------------
    # create a dictionary of the arrays for y_hat, y_test, etc
    
    fold_df = pd.DataFrame({
        "y_true": y_test,
        "y_pred": y_pred,
        "location": groups[test_idx]
    }, index = test_idx)

    fold_df = fold_df.join(test_metadata)
    all_predictions.append(fold_df)

predictions_df= pd.concat(all_predictions)
predictions_fp = PROJECT_ROOT / "outputs" / "nested_cv_predictions.csv"
predictions_df.to_csv(predictions_fp)


results_df = pd.DataFrame(outer_results)
results_df["f1_gap"] = results_df["train_f1"] - results_df["f1"]
results_df["f1_macro_gap"] = results_df["train_f1_macro"] - results_df["f1_macro"]
results_df["accuracy_gap"] = results_df["train_accuracy"] - results_df["accuracy"]

results_fp = PROJECT_ROOT / "outputs"/ "nested_cv_results.csv"
results_df.to_csv(results_fp)

print(f"F1 Binary:  {np.mean(outer_results['f1']):.3f} +/- {np.std(outer_results['f1']):.3f}")
print(f"F1 Macro:   {np.mean(outer_results['f1_macro']):.3f} +/- {np.std(outer_results['f1_macro']):.3f}")
print(f"Accuracy:   {np.mean(outer_results['accuracy']):.3f} +/- {np.std(outer_results['accuracy']):.3f}")

KeyboardInterrupt: 